# Live Detection Viewer — Real-time Vision System

**Purpose:** See what the robot sees in real-time with all detection overlays.

This notebook provides:
- Live camera feed with detection overlays
- Real-time ball, obstacle, and basket detection
- FPS counter and statistics
- Toggle controls for different detection layers
- Screenshot capture

## 1. Setup

In [ ]:
import sys
import os
import time
import yaml
import cv2
import numpy as np
from IPython.display import display, Image
import ipywidgets as widgets
import threading

project_root = '/home/steve/Notebooks/Projects/AI & Robotics Hackathon Berlin/ITQ_HackLab_Team_2'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from hardware.camera import CameraController
from perception.ball_detector import BallDetector
from perception.obstacle_detector import ObstacleDetector
from perception.basket_detector import BasketDetector

print("✓ Modules imported")

## 2. Initialize Camera and Detectors

In [ ]:
# Load config
config_path = os.path.join(project_root, 'config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Initialize camera
camera = CameraController(config)
if not camera.initialize():
    raise Exception("Camera initialization failed!")

camera.center()
camera.look_down()

# Initialize detectors
ball_detector = BallDetector(config)
obstacle_detector = ObstacleDetector(config)
basket_detector = BasketDetector(config)

print("✓ Camera and detectors initialized")
print(f"  Camera: {camera.camera_source}")
print(f"  Resolution: {camera.width}x{camera.height}")

## 3. Live Detection Viewer with Controls

In [ ]:
# Create widgets
image_widget = widgets.Image(format='jpeg', width=640, height=480)
fps_label = widgets.HTML(value="<b>FPS:</b> 0.0")
status_label = widgets.HTML(value="<b>Status:</b> Ready")

# Detection statistics
ball_count_label = widgets.HTML(value="<b>Balls:</b> 0")
boundary_label = widgets.HTML(value="<b>Boundary:</b> NO")
obstacle_label = widgets.HTML(value="<b>Obstacle:</b> NO")
basket_label = widgets.HTML(value="<b>Basket:</b> NO")

# Toggle switches for detection layers
show_balls = widgets.Checkbox(value=True, description='Show Balls')
show_obstacles = widgets.Checkbox(value=True, description='Show Obstacles')
show_basket = widgets.Checkbox(value=True, description='Show Basket')
show_rois = widgets.Checkbox(value=True, description='Show ROIs')

# Control variables
running = False
latest_frame = None
latest_detections = {
    'balls': [],
    'obstacle': {},
    'basket': {}
}

def update_feed():
    global running, latest_frame, latest_detections
    
    last_time = time.time()
    frame_count = 0
    
    while running:
        frame = camera.read()
        if frame is None:
            time.sleep(0.1)
            continue
        
        # Run detections
        balls = ball_detector.detect(frame)
        obstacle = obstacle_detector.detect_combined(frame)
        basket = basket_detector.detect(frame)
        
        # Store for screenshot
        latest_frame = frame.copy()
        latest_detections = {
            'balls': balls,
            'obstacle': obstacle,
            'basket': basket
        }
        
        # Draw overlays based on toggles
        overlay = frame.copy()
        
        if show_balls.value:
            overlay = ball_detector.draw_detections(overlay, balls)
        
        if show_obstacles.value:
            overlay = obstacle_detector.draw_detections(overlay, obstacle)
        
        if show_basket.value:
            overlay = basket_detector.draw_detection(overlay, basket)
        
        if show_rois.value:
            # Draw ROI boundaries
            h, w = overlay.shape[:2]
            roi_top = int(h * (1 - obstacle_detector.bottom_roi_frac))
            cv2.line(overlay, (0, roi_top), (w, roi_top), (0, 255, 255), 1)
            
            front_top = int(h * obstacle_detector.front_roi_top_frac)
            front_bottom = int(h * obstacle_detector.front_roi_bottom_frac)
            cv2.rectangle(overlay, (0, front_top), (w, front_bottom), (255, 0, 255), 1)
        
        # Update image
        _, buffer = cv2.imencode('.jpg', overlay)
        image_widget.value = buffer.tobytes()
        
        # Update statistics
        ball_count_label.value = f"<b>Balls:</b> {len(balls)}"
        
        if obstacle['boundary_detected']:
            boundary_label.value = f"<b style='color:red'>Boundary:</b> YES ({obstacle['turn_direction']})"
        else:
            boundary_label.value = "<b>Boundary:</b> NO"
        
        if obstacle['obstacle_detected']:
            obstacle_label.value = f"<b style='color:orange'>Obstacle:</b> YES ({obstacle['turn_direction']})"
        else:
            obstacle_label.value = "<b>Obstacle:</b> NO"
        
        if basket['basket_found']:
            basket_label.value = f"<b style='color:green'>Basket:</b> FOUND ({basket['distance']:.0f}cm)"
        else:
            basket_label.value = "<b>Basket:</b> NO"
        
        # Calculate FPS
        frame_count += 1
        if frame_count % 10 == 0:
            current_time = time.time()
            fps = 10 / (current_time - last_time)
            fps_label.value = f"<b>FPS:</b> {fps:.1f}"
            last_time = current_time
        
        time.sleep(0.033)  # ~30 FPS

def start_feed(b):
    global running
    if not running:
        running = True
        status_label.value = "<b style='color:green'>Status:</b> Running"
        thread = threading.Thread(target=update_feed)
        thread.daemon = True
        thread.start()

def stop_feed(b):
    global running
    running = False
    status_label.value = "<b style='color:red'>Status:</b> Stopped"

def capture_screenshot(b):
    global latest_frame, latest_detections
    if latest_frame is not None:
        # Draw all detections
        screenshot = latest_frame.copy()
        screenshot = ball_detector.draw_detections(screenshot, latest_detections['balls'])
        screenshot = obstacle_detector.draw_detections(screenshot, latest_detections['obstacle'])
        screenshot = basket_detector.draw_detection(screenshot, latest_detections['basket'])
        
        # Save
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        filename = f"detection_screenshot_{timestamp}.jpg"
        filepath = os.path.join(project_root, 'logs', filename)
        cv2.imwrite(filepath, screenshot)
        
        status_label.value = f"<b style='color:blue'>Screenshot saved:</b> {filename}"

# Create buttons
start_button = widgets.Button(description="▶ Start", button_style='success', icon='play')
stop_button = widgets.Button(description="⏹ Stop", button_style='danger', icon='stop')
screenshot_button = widgets.Button(description="📷 Screenshot", button_style='info', icon='camera')

start_button.on_click(start_feed)
stop_button.on_click(stop_feed)
screenshot_button.on_click(capture_screenshot)

# Layout
controls = widgets.HBox([start_button, stop_button, screenshot_button])
toggles = widgets.HBox([show_balls, show_obstacles, show_basket, show_rois])
stats = widgets.VBox([
    widgets.HBox([fps_label, status_label]),
    widgets.HBox([ball_count_label, boundary_label, obstacle_label, basket_label])
])

display(widgets.VBox([
    widgets.HTML("<h2>Live Detection Viewer</h2>"),
    controls,
    toggles,
    stats,
    image_widget
]))

## 4. Detection Details Panel

View detailed information about current detections.

In [ ]:
details_output = widgets.Output()

def update_details(b):
    global latest_detections
    with details_output:
        details_output.clear_output(wait=True)
        
        print("=" * 60)
        print("CURRENT DETECTION DETAILS")
        print("=" * 60)
        
        # Balls
        balls = latest_detections.get('balls', [])
        print(f"\n🔵 BALLS DETECTED: {len(balls)}")
        if balls:
            for i, ball in enumerate(balls, 1):
                color, centroid, distance, area = ball
                print(f"  {i}. {color.upper()}")
                print(f"     Position: {centroid}")
                print(f"     Distance: {distance:.1f} cm")
                print(f"     Area: {area:.0f} pixels²")
        
        # Obstacles
        obstacle = latest_detections.get('obstacle', {})
        print(f"\n⚠️  OBSTACLES:")
        print(f"  Boundary: {'YES' if obstacle.get('boundary_detected') else 'NO'}")
        if obstacle.get('boundary_detected'):
            print(f"    Yellow pixels: {obstacle.get('yellow_pixels', 0)}")
            print(f"    Turn direction: {obstacle.get('turn_direction', 'none')}")
        
        print(f"  Front obstacle: {'YES' if obstacle.get('obstacle_detected') else 'NO'}")
        if obstacle.get('obstacle_detected'):
            print(f"    Edge pixels: {obstacle.get('edge_pixels', 0)}")
            print(f"    Turn direction: {obstacle.get('turn_direction', 'none')}")
        
        print(f"  Priority: {obstacle.get('priority', 'none')}")
        
        # Basket
        basket = latest_detections.get('basket', {})
        print(f"\n🗑️  BASKET:")
        print(f"  Found: {'YES' if basket.get('basket_found') else 'NO'}")
        if basket.get('basket_found'):
            print(f"  Position: {basket.get('centroid')}")
            print(f"  Distance: {basket.get('distance', 0):.1f} cm")
            print(f"  Bearing: {basket.get('bearing', 0):.1f}°")
            print(f"  Aligned: {basket.get('aligned', False)}")

details_button = widgets.Button(description="📊 Show Details", button_style='primary')
details_button.on_click(update_details)

display(details_button)
display(details_output)

## 5. Camera Control Panel

In [ ]:
# Pan/Tilt sliders
pan_slider = widgets.IntSlider(
    value=0, min=-90, max=90, step=5,
    description='Pan:', continuous_update=False
)
tilt_slider = widgets.IntSlider(
    value=-30, min=-60, max=60, step=5,
    description='Tilt:', continuous_update=False
)

def on_pan_change(change):
    camera.set_pan(change['new'])

def on_tilt_change(change):
    camera.set_tilt(change['new'])

pan_slider.observe(on_pan_change, names='value')
tilt_slider.observe(on_tilt_change, names='value')

# Preset buttons
def center_camera(b):
    pan_slider.value = 0
    tilt_slider.value = 0

def look_down(b):
    pan_slider.value = 0
    tilt_slider.value = -30

center_btn = widgets.Button(description="Center", button_style='info')
down_btn = widgets.Button(description="Look Down", button_style='info')

center_btn.on_click(center_camera)
down_btn.on_click(look_down)

display(widgets.VBox([
    widgets.HTML("<h3>Camera Control</h3>"),
    pan_slider,
    tilt_slider,
    widgets.HBox([center_btn, down_btn])
]))

## 6. Cleanup

In [ ]:
# Stop feed
running = False
time.sleep(0.5)

# Release camera
camera.release()
print("✓ Cleanup complete")

## Summary

You've learned:
- ✓ Real-time vision system performance
- ✓ How multiple detectors work together
- ✓ Visual feedback interpretation
- ✓ Detection reliability and FPS
- ✓ Interactive camera control

**Next:** Move to `04_vision_control_interactive.ipynb` to control the robot based on what you see!